# Step 8 - Implicit bias detection: gender-coded word scoring (TF-IDF) vs a fine-tuned DistilBERT classifier

**Aim.** Detect *implicit* gender-coded wording (taxonomy v0.3 codes I1 masculine-coded and I2 feminine-coded) in the PESO job advertisements, compute a **bias score from -1 to +1** for each advertisement, and **flag the phrases** that carry it. Two method families are compared:

| | Method | Learns from labels? |
|---|---|---|
| **A1** | Gender-coded lexicon (unweighted counts) | no |
| **A2** | Gender-coded lexicon with TF-IDF weights and a tuned evidence threshold | threshold only |
| **B** | TF-IDF (1-2 grams) + logistic regression | yes |
| **C** | Fine-tuned DistilBERT multi-label classifier (3 seeds) | yes |

**Standing conditions for this step.** Same test set as Step 7 (400 ads). No human annotation (handled in separate work). No mock data. Every number is **agreement with rule-derived (silver) I1/I2 labels**, not validity. Silver I1/I2 labels were produced from a word list in Steps 4-5, so the lexicon systems are circular by construction; this is stated at every point where it matters. Nothing here changes Steps 1-7 or the open taxonomy v0.3 decisions.

In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
import json, re, time, random, copy, math
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 230)

SMOKE = os.environ.get("S8_SMOKE") == "1"
OUT = Path("outputs") / ("s8_implicit_detection_smoke" if SMOKE else "s8_implicit_detection")
OUT.mkdir(parents=True, exist_ok=True)
S7 = Path("outputs") / "s7_explicit_detection"

PARAMS = {
    "test_sample": "the 400 ads of Step 7 (seed 20261026, strata = Step 5b route), read from outputs/s7_explicit_detection/test_sample_400_ads.csv",
    "val_share_of_pool": 0.10, "val_seed": 42,
    "reference": "silver I1/I2 spans of Step 5b (taxonomy v0.3), valid offsets only",
    "lexicon_stems": "Gaucher-style stem lists retained from Step 6/8c work (equal-step8e-layerA-rule-detection.ipynb); 'confidential*' excluded",
    "tfidf_token_pattern": r"[\w'-]+", "tfidf_sublinear_tf": True, "tfidf_weight": "(1 + ln tf) * idf, divided by the L2 norm of the unit vector; idf fitted on training units only",
    "tau_grid": [0.0] + [round(float(x), 2) for x in np.arange(0.05, 0.95, 0.05)],
    "lr_ngram": [1, 2], "lr_min_df": 2, "lr_C_grid": [0.1, 1, 10, 100], "lr_class_weight": "balanced",
    "threshold_grid": [round(float(x), 2) for x in np.arange(0.05, 0.951, 0.05)],
    "phrase_B": {"min_share_of_top_contribution": 0.25, "max_words_per_unit_label": 3},
    "model": "distilbert/distilbert-base-uncased", "head": "2 sigmoid outputs (masculine, feminine), BCE loss",
    "seeds": [1, 2, 3], "epochs": 4, "batch": 32, "lr": 5e-5, "weight_decay": 0.01, "warmup_share": 0.10, "max_len": 96,
    "epoch_selection": "best validation F1 (implicit = either label, threshold 0.5)",
    "threshold_selection": "best validation F1 over threshold_grid",
    "phrase_C": {"method": "word occlusion on flagged test units", "candidate_cutoff": "min(0.3, tuned threshold)", "min_drop": 0.10, "min_share_of_top_drop": 0.5, "max_words_per_unit_label": 3, "report_rule": "phrase kept if flagged in at least 2 of 3 seeds"},
    "bootstrap_resamples": 1000, "bootstrap_seed": 7, "ci": "95% percentile, ads resampled with replacement",
    "phrase_match_rule": "predicted phrase overlaps a silver I1/I2 span by at least one character",
}
if SMOKE:
    PARAMS.update({"seeds": [1], "epochs": 1, "train_subsample_units": 300})
json.dump(PARAMS, open(OUT / "parameters.json", "w"), indent=1)
print("SMOKE RUN" if SMOKE else "FULL RUN")
print(json.dumps({k: v for k, v in PARAMS.items() if k not in ("tau_grid", "threshold_grid")}, indent=1))

rd = lambda p: pd.read_csv(p, encoding="utf-8-sig")
units = rd("outputs/analysis/units.csv")
ad_ctx = rd("outputs/s5b_taxonomy_test/ad_context_v0.3.csv")
sp_all = rd("outputs/s5b_taxonomy_test/preannotation_spans_v0.3.csv")
ad_ctx["route_v03"] = ad_ctx["route_v03"].fillna("no span")
print(f"{units['record_id'].nunique():,} ads, {len(units):,} units, {len(sp_all):,} v0.3 spans")

FULL RUN
{
 "test_sample": "the 400 ads of Step 7 (seed 20261026, strata = Step 5b route), read from outputs/s7_explicit_detection/test_sample_400_ads.csv",
 "val_share_of_pool": 0.1,
 "val_seed": 42,
 "reference": "silver I1/I2 spans of Step 5b (taxonomy v0.3), valid offsets only",
 "lexicon_stems": "Gaucher-style stem lists retained from Step 6/8c work (equal-step8e-layerA-rule-detection.ipynb); 'confidential*' excluded",
 "tfidf_token_pattern": "[\\w'-]+",
 "tfidf_sublinear_tf": true,
 "tfidf_weight": "(1 + ln tf) * idf, divided by the L2 norm of the unit vector; idf fitted on training units only",
 "lr_ngram": [
  1,
  2
 ],
 "lr_min_df": 2,
 "lr_C_grid": [
  0.1,
  1,
  10,
  100
 ],
 "lr_class_weight": "balanced",
 "phrase_B": {
  "min_share_of_top_contribution": 0.25,
  "max_words_per_unit_label": 3
 },
 "model": "distilbert/distilbert-base-uncased",
 "head": "2 sigmoid outputs (masculine, feminine), BCE loss",
 "seeds": [
  1,
  2,
  3
 ],
 "epochs": 4,
 "batch": 32,
 "lr": 5e-

## 1. Test subset and training pool

**Same test set as Step 7.** The 400 advertisements are the stratified random sample drawn in Step 7 (seed 20261026; strata = Step 5b ad route; proportional allocation, minimum 2 per stratum). Re-using it keeps the explicit (Step 7) and implicit (Step 8) results comparable on the same advertisements and avoids a second draw. The stratified sample is a design choice of this project; we found no 2022-2025 source specific to stratified sampling of job advertisements.

**Leakage control.** The 1,100 other advertisements are split by advertisement (not by unit) into 990 training and 110 validation advertisements, stratified by route with the same seed and rule as Step 7. TF-IDF vocabularies, logistic-regression weights, the DistilBERT weights, the evidence threshold and every other tuned value use only these 1,100 advertisements; the test set is scored once. Kapoor and Narayanan [14] describe leakage between training and test data as a cause of inflated, non-reproducible results in machine-learning-based science.

In [2]:
test_ads = sorted(rd(S7 / "test_sample_400_ads.csv")["record_id"])
route_of = dict(zip(ad_ctx["record_id"], ad_ctx["route_v03"]))
employer_of = units.drop_duplicates("record_id").set_index("record_id")["employer_id"].to_dict()
unit_ad = dict(zip(units["unit_id"], units["record_id"]))
text_of = dict(zip(units["unit_id"], units["text_masked"].astype(str)))

pool_ads = sorted(set(units["record_id"]) - set(test_ads))
rng_split = np.random.RandomState(PARAMS["val_seed"])
pool_route = {a: route_of[a] for a in pool_ads}
val_ads = set()
for k in sorted(set(pool_route.values())):
    ids = sorted(a for a, r in pool_route.items() if r == k)
    n_val = max(1, int(round(len(ids) * PARAMS["val_share_of_pool"])))
    val_ads |= set(rng_split.choice(ids, n_val, replace=False))
train_ads = sorted(set(pool_ads) - val_ads)

tr_units = units[units["record_id"].isin(train_ads)].reset_index(drop=True)
va_units = units[units["record_id"].isin(val_ads)].reset_index(drop=True)
test_units = units[units["record_id"].isin(test_ads)].reset_index(drop=True)
assert not (set(train_ads) | val_ads) & set(test_ads)
print(f"train {len(train_ads)} ads / {len(tr_units):,} units; val {len(val_ads)} ads / {len(va_units):,} units; test {len(test_ads)} ads / {len(test_units):,} units")

train 990 ads / 5,253 units; val 110 ads / 576 units; test 400 ads / 2,142 units


## 2. Task, reference labels and the bias score

**Reference (silver) labels.** Silver I1 (masculine-coded) and I2 (feminine-coded) spans of Step 5b with valid offsets. A **unit** (sentence/requirement) is implicit-positive if it holds at least one I1 or I2 span; direction labels are *masculine* (holds I1) and *feminine* (holds I2), and a unit can hold both. An **advertisement** is implicit-positive if any of its units is.

**Circularity, stated plainly.** The silver I1/I2 spans come from the same coded-word stems that system A uses. System A is therefore expected to agree almost perfectly, and agreement does not show that the flagged words signal gender to applicants. Our methods file already notes that the most frequent hits are duty words (*analytical, leadership, interpersonal, honest, responsible*) that may describe the job, and that the Gaucher word list was designed for US-English advertisements and has not been tested on Philippine advertisements. Jiang et al. [8] argue that lexicons need local adaptation, and Hu et al. [4] report that implicit traits remain associated with gender even when explicit gender words are removed. For these reasons results are reported as **agreement with silver labels** and disagreements are listed for review, not resolved.

**Bias score (-1 to +1).** For an advertisement, let M be the evidence for masculine-coded wording and F the evidence for feminine-coded wording, summed over the advertisement's flagged units:

`score = (M - F) / (M + F)`, and `score = 0` when `M + F = 0`.

+1 means all flagged evidence is masculine-coded, -1 all feminine-coded, 0 balanced or no evidence (the advertisement label separates "balanced" from "no signal"). The evidence is a count of coded words for A1, the sum of TF-IDF weights of coded words for A2, and the sum of predicted probabilities of flagged units for B and C. The silver score uses counts of silver I1 and I2 spans. Our methods file records that Dikshit et al. [1] label an advertisement masculine-coded (agentic) or feminine-coded (communal) by which coded word type dominates; the same balance is the basis here. The paper's results were not verified against the full text. **The score measures the balance of coded language, not harm:** a score near 0 does not mean an advertisement is unbiased, and explicit bias is covered by Step 7, not here.

In [3]:
valid = sp_all["span_offset_valid"].astype(str).str.lower().eq("true")
imp = sp_all[sp_all["expression_code"].isin(["I1", "I2"]) & valid].drop_duplicates(["unit_id", "span_start", "span_end", "expression_code"]).copy()
imp["review_flag"] = imp["review_flag"].fillna("")
print(f"silver I1/I2 spans with valid offsets: {len(imp)} of {int(sp_all['expression_code'].isin(['I1','I2']).sum())}")

gold_by_unit = {}
bad_offsets = 0
for r in imp.itertuples():
    t = text_of.get(r.unit_id, "")
    if t[int(r.span_start):int(r.span_end)].lower() != str(r.span_text).lower():
        bad_offsets += 1
    gold_by_unit.setdefault(r.unit_id, []).append((int(r.span_start), int(r.span_end), r.expression_code, str(r.span_text).lower()))
print("silver spans that do not reproduce inside their unit text:", bad_offsets)

def labels(df):
    ym = np.array([any(g[2] == "I1" for g in gold_by_unit.get(u, [])) for u in df["unit_id"]])
    yf = np.array([any(g[2] == "I2" for g in gold_by_unit.get(u, [])) for u in df["unit_id"]])
    return ym, yf, ym | yf

ym_tr, yf_tr, y_tr = labels(tr_units); ym_va, yf_va, y_va = labels(va_units); ym_te, yf_te, y_te = labels(test_units)

def ad_pos(df, y):
    return int(df.loc[y, "record_id"].nunique())
rows = []
for name, df, ym, yf, y, ads in (("train", tr_units, ym_tr, yf_tr, y_tr, train_ads), ("validation", va_units, ym_va, yf_va, y_va, sorted(val_ads)), ("test", test_units, ym_te, yf_te, y_te, test_ads)):
    n_spans = sum(len(gold_by_unit.get(u, [])) for u in df["unit_id"])
    rows.append({"split": name, "ads": len(ads), "units": len(df), "units implicit": int(y.sum()), "units masculine (I1)": int(ym.sum()), "units feminine (I2)": int(yf.sum()),
                 "units with both": int((ym & yf).sum()), "ads implicit": ad_pos(df, y), "silver spans": n_spans, "unit prevalence": round(float(y.mean()), 3)})
split_tbl = pd.DataFrame(rows).set_index("split")
display(split_tbl)
split_tbl.to_csv(OUT / "split_summary.csv", encoding="utf-8-sig")

rolenoun = imp["review_flag"].eq("P_candidate_role_noun")
print(f"silver I1/I2 spans that Step 5 also flagged as a possible polysemantic role noun (P candidate): {int(rolenoun.sum())} of {len(imp)} ({rolenoun.mean():.1%})")

silver I1/I2 spans with valid offsets: 728 of 729
silver spans that do not reproduce inside their unit text: 0


,ads,units,units implicit,units masculine (I1),units feminine (I2),units with both,ads implicit,silver spans,unit prevalence
split,,,,,,,,,
train,990,5253,404,235,205,36,277,497,0.077
validation,110,576,36,13,28,5,27,47,0.062
test,400,2142,155,76,90,11,109,184,0.072


silver I1/I2 spans that Step 5 also flagged as a possible polysemantic role noun (P candidate): 106 of 728 (14.6%)


## 3. Method A and B: gender-coded word scoring and TF-IDF

**A1: gender-coded lexicon.** Word stems for masculine-coded (agentic) and feminine-coded (communal) wording follow the Gaucher-style lists retained from earlier project work, matched by prefix on whole words. Gaucher, Friesen and Kay (2011) is the origin of the word lists; it is outside the 2022-2025 window and is cited only to identify them, and the project's lists are the Gender Decoder implementation, not the paper's table. **One designed difference from the silver labels:** `confidential*` is excluded because it shares the `confident` stem and describes information handling, as recorded in the project methods file. The silver labels still contain these words, so they will be counted as disagreements.

**Why a lexicon is the baseline.** Frissen et al. [25] tag masculine-coded and feminine-coded language as separate categories in real job advertisements, and Everitt et al. [23] use feminine and masculine as two labels in an eight-label scheme, so a masculine/feminine split is an established framing. [23] also note that rule or find-and-replace approaches handle predictable terms but do not scale because words are polysemous, which is the weakness the next methods are meant to test.

**A2: TF-IDF-weighted lexicon.** Coded words that occur in almost every unit (*support, responsible*) carry less evidence than rare ones. Each matched word gets the weight (1 + ln tf) x idf, divided by the L2 norm of the unit's TF-IDF vector, with idf fitted on training units only (an unseen word gets the maximum idf). A unit is flagged when its total coded weight is at least tau; **tau is chosen on the validation set** from the grid in `parameters.json` (tau = 0 means any match, which equals A1). We found no 2022-2025 source that validates TF-IDF weighting of coded-word counts for job advertisements, so this is our own design and is tested, not assumed.

**B: TF-IDF + logistic regression.** Unigram and bigram TF-IDF features (min_df 2, sublinear tf) feed two logistic regressions (masculine, feminine) with balanced class weights. C and the decision threshold are chosen on validation F1. Jiang et al. [8] argue that a lexicon needs adaptation to local text; a model trained on local text is one data-driven way to adapt it, but here the labels come from the lexicon, so B can learn only what the labels contain. **Flagged phrases for B** are the unigrams with the largest positive contribution (coefficient x TF-IDF value) to a flagged label, keeping words at least 25% of the unit's top contribution, at most 3 per label.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

MASC = ("active", "adventurous", "aggress", "ambitio", "analy", "assert", "athlet", "autonom", "boast", "challeng", "compet", "confident", "courag",
        "decisive", "decision", "determin", "dominan", "driven", "fearless", "force", "headstrong", "hierarch", "impulsive", "independen", "individual",
        "intellect", "lead", "logic", "objective", "opinion", "outspoken", "persist", "self-relian", "self-sufficien", "stubborn", "superior", "unreasonab")
FEM = ("agree", "affectionate", "child", "cheer", "collab", "commit", "communal", "compassion", "connect", "considerate", "cooperat", "depend", "emotiona",
       "empath", "feel", "flatterable", "gentle", "honest", "interper", "interdependen", "kind", "kinship", "loyal", "modest", "nurtur", "pleasant",
       "polite", "quiet", "respons", "sensitiv", "submissive", "support", "sympath", "tender", "together", "trust", "understand", "warm", "enthusias", "yield", "share")
EXCLUDE = re.compile(r"confidential", re.I)
def stem_rx(stems):
    return re.compile(r"(?<!\w)(?:" + "|".join(re.escape(s) for s in sorted(stems, key=len, reverse=True)) + r")[a-z-]*(?!\w)", re.I)
RX = {"I1": stem_rx(MASC), "I2": stem_rx(FEM)}

def lex_matches(text):
    ms = []
    for c, rx in RX.items():
        for m in rx.finditer(text):
            if not EXCLUDE.match(m.group(0)):
                ms.append((m.start(), m.end(), c, m.group(0).lower()))
    ms.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    out, last = [], -1
    for m in ms:
        if m[0] >= last:
            out.append(m); last = m[1]
    return out
LEX_M = {u: lex_matches(t) for u, t in text_of.items()}

TOK = re.compile(PARAMS["tfidf_token_pattern"])
FIT_IDS_A = set(tr_units["unit_id"])
tf_a = TfidfVectorizer(lowercase=True, token_pattern=PARAMS["tfidf_token_pattern"], sublinear_tf=True)
tf_a.fit(tr_units["text_masked"].astype(str))
IDF = dict(zip(tf_a.get_feature_names_out(), tf_a.idf_)); IDF_MAX = float(tf_a.idf_.max())

def unit_weights(text):
    c = Counter(TOK.findall(text.lower()))
    v = {w: (1 + math.log(n)) * IDF.get(w, IDF_MAX) for w, n in c.items()}
    return v, (math.sqrt(sum(x * x for x in v.values())) or 1.0)

def lex_eval(df, weighted):
    n = len(df); M = np.zeros(n); F = np.zeros(n); W = [[] for _ in range(n)]
    for i, (uid, text) in enumerate(zip(df["unit_id"], df["text_masked"].astype(str))):
        ms = LEX_M[uid]
        if not ms:
            continue
        v, norm = unit_weights(text) if weighted else ({}, 1.0)
        for m in ms:
            wt = (v.get(m[3], IDF_MAX) / norm) if weighted else 1.0
            W[i].append((m, wt))
            if m[2] == "I1": M[i] += wt
            else: F[i] += wt
    return M, F, W

def lex_gate(df, M, F, W, tau):
    tot = M + F; on = (tot > 0) & (tot >= tau)
    phr = [(uid, m[0], m[1], m[2], wt) for i, uid in enumerate(df["unit_id"]) if on[i] for m, wt in W[i]]
    return {"fm": on & (M > 0), "ff": on & (F > 0), "M": np.where(on, M, 0.0), "F": np.where(on, F, 0.0), "phr": phr}

def f1_bin(y, p):
    tp = int((y & p).sum()); fp = int((~y & p).sum()); fn = int((y & ~p).sum())
    return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0

# A1 on test; A2 tau on validation
Mt, Ft, Wt = lex_eval(test_units, False)
RUN_A1 = lex_gate(test_units, Mt, Ft, Wt, 0.0)
Mv, Fv, Wv = lex_eval(va_units, True)
tau_rows = [{"tau": t, "val_F1": f1_bin(y_va, lex_gate(va_units, Mv, Fv, Wv, t)["fm"] | lex_gate(va_units, Mv, Fv, Wv, t)["ff"])} for t in PARAMS["tau_grid"]]
tau_df = pd.DataFrame(tau_rows)
BEST_TAU = float(tau_df.loc[tau_df["val_F1"].idxmax(), "tau"])
display(tau_df.round(4).T)
print("tau selected on validation:", BEST_TAU)
Mt2, Ft2, Wt2 = lex_eval(test_units, True)
RUN_A2 = lex_gate(test_units, Mt2, Ft2, Wt2, BEST_TAU)
tau_df.to_csv(OUT / "tau_grid_validation.csv", index=False)

# lexicon vs silver on the training pool (circularity check)
pool_df = pd.concat([tr_units, va_units])
lex_spans = {(u, m[0], m[1]) for u in pool_df["unit_id"] for m in LEX_M[u]}
sil_spans = {(u, g[0], g[1]) for u in pool_df["unit_id"] for g in gold_by_unit.get(u, [])}
only_sil = sil_spans - lex_spans; only_lex = lex_spans - sil_spans
print(f"pool: lexicon spans {len(lex_spans)}, silver spans {len(sil_spans)}, shared {len(lex_spans & sil_spans)}; silver-only {len(only_sil)}, lexicon-only {len(only_lex)}")
print("silver-only wording (top 8):", Counter(text_of[u][s:e].lower() for u, s, e in only_sil).most_common(8))
print("lexicon-only wording (top 8):", Counter(text_of[u][s:e].lower() for u, s, e in only_lex).most_common(8))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
tau,0.0,0.05,0.1,0.15,0.2000,0.2500,0.3000,0.3500,0.4000,0.4500,0.5000,0.5500,0.6000,0.6500,0.7,0.75,0.8000,0.8500,0.9000
val_F1,1.0,1.00,1.0,1.00,0.9859,0.9714,0.9565,0.9412,0.8571,0.7143,0.5882,0.5306,0.4348,0.2857,0.2,0.20,0.1538,0.1538,0.1538


tau selected on validation: 0.0
pool: lexicon spans 533, silver spans 544, shared 533; silver-only 11, lexicon-only 0
silver-only wording (top 8): [('confidentiality', 6), ('confidential', 5)]
lexicon-only wording (top 8): []


In [5]:
# Method B: TF-IDF 1-2 grams + logistic regression (fit on training units only)
FIT_IDS_B = set(tr_units["unit_id"])
tr_fit = tr_units
if SMOKE:
    sub = np.random.RandomState(0).permutation(len(tr_units))[:PARAMS["train_subsample_units"]]
vec_b = TfidfVectorizer(lowercase=True, ngram_range=tuple(PARAMS["lr_ngram"]), min_df=PARAMS["lr_min_df"], sublinear_tf=True, token_pattern=PARAMS["tfidf_token_pattern"])
Xtr = vec_b.fit_transform(tr_units["text_masked"].astype(str)); Xva = vec_b.transform(va_units["text_masked"].astype(str)); Xte = vec_b.transform(test_units["text_masked"].astype(str))
FEAT = vec_b.get_feature_names_out()
print(f"vocabulary: {len(FEAT):,} features ({sum(' ' not in f for f in FEAT):,} unigrams)")

models, rows, best = {}, [], None
for C in PARAMS["lr_C_grid"]:
    lm = LogisticRegression(C=C, class_weight=PARAMS["lr_class_weight"], max_iter=2000).fit(Xtr, ym_tr)
    lf = LogisticRegression(C=C, class_weight=PARAMS["lr_class_weight"], max_iter=2000).fit(Xtr, yf_tr)
    models[C] = (lm, lf)
    pm, pf = lm.predict_proba(Xva)[:, 1], lf.predict_proba(Xva)[:, 1]
    for th in PARAMS["threshold_grid"]:
        f = f1_bin(y_va, (pm >= th) | (pf >= th))
        rows.append({"C": C, "threshold": th, "val_F1": f})
        if best is None or f > best[0] + 1e-12:
            best = (f, C, th)
B_grid = pd.DataFrame(rows)
display(B_grid.pivot(index="C", columns="threshold", values="val_F1").round(3))
B_VALF1, B_C, B_TH = best
print(f"selected on validation: C={B_C}, threshold={B_TH}, validation F1={B_VALF1:.4f}")
B_grid.to_csv(OUT / "logreg_grid_validation.csv", index=False)

lm, lf = models[B_C]
pm_te, pf_te = lm.predict_proba(Xte)[:, 1], lf.predict_proba(Xte)[:, 1]

def b_run():
    fm, ff = pm_te >= B_TH, pf_te >= B_TH
    phr, seen = [], set()
    for i, (uid, text) in enumerate(zip(test_units["unit_id"], test_units["text_masked"].astype(str))):
        if not (fm[i] or ff[i]):
            continue
        row = Xte[i]
        for flag, coef, c in ((fm[i], lm.coef_[0], "I1"), (ff[i], lf.coef_[0], "I2")):
            if not flag:
                continue
            contrib = {FEAT[j]: coef[j] * x for j, x in zip(row.indices, row.data) if " " not in FEAT[j] and coef[j] * x > 0}
            if not contrib:
                continue
            mx = max(contrib.values())
            keep = sorted([w for w, v in contrib.items() if v >= PARAMS["phrase_B"]["min_share_of_top_contribution"] * mx], key=lambda w: -contrib[w])[:PARAMS["phrase_B"]["max_words_per_unit_label"]]
            for m in TOK.finditer(text):
                w = m.group(0).lower()
                if w in keep and (uid, m.start(), m.end(), c) not in seen:
                    seen.add((uid, m.start(), m.end(), c)); phr.append((uid, m.start(), m.end(), c, float(contrib[w])))
    return {"fm": fm, "ff": ff, "M": np.where(fm, pm_te, 0.0), "F": np.where(ff, pf_te, 0.0), "phr": phr}
RUN_B = b_run()
print(f"B on test: {int((RUN_B['fm'] | RUN_B['ff']).sum())} flagged units, {len(RUN_B['phr'])} flagged phrases")

vocabulary: 5,162 features (1,722 unigrams)


threshold,0.05,0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90,0.95
C,,,,,,,,,,,,,,,,,,,
0.1,0.118,0.118,0.125,0.137,0.163,0.235,0.355,0.421,0.526,0.674,0.737,0.646,0.644,0.593,0.500,0.400,0.200,0.154,0.054
1.0,0.146,0.247,0.362,0.449,0.496,0.586,0.673,0.736,0.765,0.827,0.849,0.829,0.794,0.818,0.800,0.712,0.667,0.500,0.435
10.0,0.400,0.574,0.708,0.762,0.759,0.779,0.800,0.833,0.857,0.857,0.841,0.853,0.836,0.848,0.848,0.862,0.862,0.800,0.737
100.0,0.687,0.756,0.775,0.789,0.784,0.795,0.817,0.817,0.853,0.853,0.853,0.853,0.866,0.862,0.862,0.862,0.862,0.875,0.857


selected on validation: C=100, threshold=0.9, validation F1=0.8750
B on test: 116 flagged units, 215 flagged phrases


## 4. Method C: fine-tuned DistilBERT multi-label classifier

**Model and head.** `distilbert/distilbert-base-uncased` (Sanh et al., 2019; outside the 2022-2025 window, cited only to identify the model and not verified in this session) with two sigmoid outputs, one for masculine-coded and one for feminine-coded wording, trained with binary cross-entropy. Two independent outputs are used because a unit can carry both directions and because Everitt et al. [23] frame bias detection as multi-label classification. Everitt et al. [23] also report that fine-tuned models outperformed prompting for implicit bias detection (abstract-level reading), which is why a fine-tuned encoder, not a prompted model, is compared with the lexicon here. DistilBERT is the smaller of the two cached encoders and was the model used in Step 6.

**Input and labels.** One unit (sentence/requirement) per example, maximum 96 tokens. Labels are the unit-level silver masculine and feminine labels from section 2. The full advertisement is not used as context at this step.

**Training parameters.** 3 seeds (1, 2, 3); 4 epochs; batch 32; learning rate 5e-5 with linear decay and 10% warm-up; weight decay 0.01; gradient clipping 1.0. The epoch is chosen by validation F1 at threshold 0.5 and the decision threshold is then tuned on validation probabilities. Test probabilities are stored for the selected epoch only; the test set is never used for selection. Settings are standard defaults and were not otherwise tuned.

**Flagged phrases for C.** DistilBERT classifies whole units, so phrases are found by **word occlusion**: each word is removed in turn from a flagged test unit and the drop in the predicted probability for that label is recorded. A word is flagged when its drop is at least 0.10 and at least half of the unit's largest drop, at most 3 words per label. We found no verified 2022-2025 source for occlusion-based phrase attribution on job advertisements, so this is standard practice applied by us, not a cited method. A phrase is reported when it is flagged in at least 2 of the 3 seeds. Raza et al. [26] tag bias spans at token level (including job-hiring text); here the model is trained on units, so its phrase flags are an attribution and not a trained span tagger.

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

MODEL = PARAMS["model"]; SEEDS = PARAMS["seeds"]; EPOCHS = PARAMS["epochs"]; BATCH = PARAMS["batch"]; MAXLEN = PARAMS["max_len"]
tok = AutoTokenizer.from_pretrained(MODEL)
enc_ids = lambda df: tok(df["text_masked"].astype(str).tolist(), truncation=True, max_length=MAXLEN)["input_ids"]
ids_tr, ids_va, ids_te = enc_ids(tr_units), enc_ids(va_units), enc_ids(test_units)
Y_tr = np.stack([ym_tr, yf_tr], 1).astype(np.float32)
lens = [len(x) for x in ids_tr + ids_va + ids_te]
print(f"tokens per unit: median {int(np.median(lens))}, max {max(lens)}; units at the {MAXLEN}-token limit: {sum(l >= MAXLEN for l in lens)}")
train_idx = np.arange(len(ids_tr))
if SMOKE:
    train_idx = np.random.RandomState(0).permutation(len(ids_tr))[:PARAMS["train_subsample_units"]]

def collate(rows):
    m = max(len(r) for r in rows)
    ids = torch.full((len(rows), m), tok.pad_token_id); att = torch.zeros((len(rows), m), dtype=torch.long)
    for i, r in enumerate(rows):
        ids[i, :len(r)] = torch.tensor(r); att[i, :len(r)] = 1
    return ids, att

@torch.no_grad()
def probs(model, ids_list, bs=64):
    model.eval(); out = []
    order = np.argsort([len(x) for x in ids_list]); 
    for i in range(0, len(ids_list), bs):
        b = [ids_list[j] for j in order[i:i + bs]]
        ids, att = collate(b); out.append(torch.sigmoid(model(input_ids=ids, attention_mask=att).logits).numpy())
    p = np.concatenate(out); res = np.zeros_like(p); res[order] = p
    return res

def occlusion(model, text, base):
    words = [(m.start(), m.end()) for m in TOK.finditer(text)]
    if not words:
        return []
    variants = [text[:s] + text[e:] for s, e in words]
    p = probs(model, tok(variants, truncation=True, max_length=MAXLEN)["input_ids"])
    return [[s, e, float(base[0] - p[i, 0]), float(base[1] - p[i, 1])] for i, (s, e) in enumerate(words)]

def train_one(seed):
    cache = OUT / f"distilbert_seed{seed}.json"
    if cache.exists():
        print(f"seed {seed}: loaded cached result"); return json.load(open(cache))
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2, problem_type="multi_label_classification")
    opt = torch.optim.AdamW(model.parameters(), lr=PARAMS["lr"], weight_decay=PARAMS["weight_decay"])
    n = len(train_idx); steps = EPOCHS * math.ceil(n / BATCH)
    sched = get_linear_schedule_with_warmup(opt, int(PARAMS["warmup_share"] * steps), steps)
    lossf = torch.nn.BCEWithLogitsLoss(); g = np.random.RandomState(seed)
    best, log = None, []
    for ep in range(1, EPOCHS + 1):
        model.train(); perm = train_idx[g.permutation(n)]; t0 = time.time(); tot = 0.0
        for i in range(0, n, BATCH):
            b = perm[i:i + BATCH]; ids, att = collate([ids_tr[j] for j in b])
            loss = lossf(model(input_ids=ids, attention_mask=att).logits, torch.tensor(Y_tr[b]))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad(); tot += loss.item() * len(b)
        pv = probs(model, ids_va); f1v = f1_bin(y_va, pv.max(1) >= 0.5)
        log.append({"seed": seed, "epoch": ep, "train_loss": round(tot / n, 4), "val_F1_at_0.5": round(f1v, 4), "seconds": round(time.time() - t0, 1)})
        print(log[-1], flush=True)
        if best is None or f1v > best["val_f1"]:
            best = {"epoch": ep, "val_f1": f1v, "pv": pv, "pt": probs(model, ids_te), "state": copy.deepcopy(model.state_dict())}
    ths = PARAMS["threshold_grid"]; fs = [f1_bin(y_va, best["pv"].max(1) >= t) for t in ths]
    thr = ths[int(np.argmax(fs))]
    model.load_state_dict(best["state"])
    cut = min(0.3, thr); occ = {}
    for i, uid in enumerate(test_units["unit_id"]):
        if best["pt"][i].max() >= cut:
            occ[uid] = occlusion(model, str(test_units["text_masked"].iloc[i]), best["pt"][i])
    res = {"seed": seed, "best_epoch": best["epoch"], "val_f1_at_0.5": best["val_f1"], "threshold": thr, "val_f1_at_threshold": float(max(fs)), "log": log,
           "test_probs": {u: [float(x) for x in best["pt"][i]] for i, u in enumerate(test_units["unit_id"])}, "occlusion": occ}
    json.dump(res, open(cache, "w")); return res

RES_C = [train_one(s) for s in SEEDS]
LOG_C = pd.DataFrame([r for res in RES_C for r in res["log"]])
display(LOG_C)
display(pd.DataFrame([{k: r[k] for k in ("seed", "best_epoch", "val_f1_at_0.5", "threshold", "val_f1_at_threshold")} for r in RES_C]))
LOG_C.to_csv(OUT / "distilbert_training_log.csv", index=False)

d:\Dev\project-equal\equal-v2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tokens per unit: median 8, max 70; units at the 96-token limit: 0
seed 1: loaded cached result
seed 2: loaded cached result
seed 3: loaded cached result


,seed,epoch,train_loss,val_F1_at_0.5,seconds
0,1,1,0.2017,0.6909,236.4
1,1,2,0.0387,0.8732,238.7
2,1,3,0.0128,0.8857,234.6
3,1,4,0.0057,0.9143,239.3
4,2,1,0.2078,0.6909,238.3
5,2,2,0.0340,0.8955,237.2
6,2,3,0.0109,0.9254,235.7
7,2,4,0.0041,0.9565,241.6
8,3,1,0.2258,0.7797,236.4
9,3,2,0.0333,0.9254,238.0


,seed,best_epoch,val_f1_at_0.5,threshold,val_f1_at_threshold
0,1,4,0.914286,0.65,0.925373
1,2,4,0.956522,0.45,0.956522
2,3,3,0.956522,0.10,0.956522


In [7]:
def distil_run(res):
    thr = res["threshold"]; P = np.array([res["test_probs"][u] for u in test_units["unit_id"]])
    fm, ff = P[:, 0] >= thr, P[:, 1] >= thr
    phr = []
    cfg = PARAMS["phrase_C"]
    for i, uid in enumerate(test_units["unit_id"]):
        if not (fm[i] or ff[i]) or uid not in res["occlusion"]:
            continue
        for flag, k, c in ((fm[i], 2, "I1"), (ff[i], 3, "I2")):
            if not flag:
                continue
            rows = res["occlusion"][uid]; mx = max(r[k] for r in rows)
            if mx < cfg["min_drop"]:
                continue
            keep = sorted([r for r in rows if r[k] >= max(cfg["min_drop"], cfg["min_share_of_top_drop"] * mx)], key=lambda r: -r[k])[:cfg["max_words_per_unit_label"]]
            phr += [(uid, int(r[0]), int(r[1]), c, float(r[k])) for r in keep]
    return {"fm": fm, "ff": ff, "M": np.where(fm, P[:, 0], 0.0), "F": np.where(ff, P[:, 1], 0.0), "phr": phr}

RUNS_C = [distil_run(r) for r in RES_C]
for r, run in zip(RES_C, RUNS_C):
    print(f"seed {r['seed']}: best epoch {r['best_epoch']}, threshold {r['threshold']}, flagged units {int((run['fm'] | run['ff']).sum())}, phrases {len(run['phr'])}")

seed 1: best epoch 4, threshold 0.65, flagged units 152, phrases 158
seed 2: best epoch 4, threshold 0.45, flagged units 153, phrases 164
seed 3: best epoch 3, threshold 0.1, flagged units 151, phrases 185


## 5. Evaluation design

**Levels.** *Unit*: implicit detection (either direction), plus masculine-only and feminine-only. *Advertisement*: any implicit unit. *Phrase*: a flagged phrase counts as correct if it overlaps a silver I1/I2 span by at least one character; *phrase + direction* also requires the same direction. Precision, recall and F1 are reported at every level; the false-positive rate (FPR = FP / (FP + TN)) only where negatives are defined (unit and advertisement levels), as in Step 7.

**Uncertainty.** 95% percentile intervals from 1,000 bootstrap resamples of the 400 advertisements (seed 7). DistilBERT is reported as the mean of 3 seeds (metrics from seed-mean counts). Paired differences use the same resamples. With few positive advertisements the intervals are wide; this is expected and is shown, not hidden. We found no verified 2022-2025 source for the bootstrap or for FPR as a bias-detection metric; both are standard practice and are used as in Step 7.

In [8]:
LEVELS = ["unit", "unit_masc", "unit_fem", "ad", "phrase", "phrase_dir"]; FPR_LEVELS = LEVELS[:4]
ad_ids = sorted(test_ads); NA = len(ad_ids); ad_idx = {a: i for i, a in enumerate(ad_ids)}
te_ad = test_units["record_id"].map(ad_idx).to_numpy()
GOLD = [(u, g[0], g[1], g[2], g[3]) for u in test_units["unit_id"] for g in gold_by_unit.get(u, [])]
G_by_unit = {}
for gi, g in enumerate(GOLD):
    G_by_unit.setdefault(g[0], []).append(gi)
G_ad = np.array([ad_idx[unit_ad[g[0]]] for g in GOLD])
print(f"test: {len(test_units):,} units, {NA} ads, {len(GOLD)} silver I1/I2 spans ({sum(g[3]=='I1' for g in GOLD)} I1, {sum(g[3]=='I2' for g in GOLD)} I2)")

def phrase_match(phr):
    seen, P = set(), []
    for uid, s, e, c, wt in phr:
        if (uid, s, e, c) not in seen:
            seen.add((uid, s, e, c)); P.append((uid, s, e, c))
    mA = np.zeros(len(GOLD)); mD = np.zeros(len(GOLD)); fpA = np.zeros(NA); fpD = np.zeros(NA)
    for uid, s, e, c in P:
        hitA = hitD = False
        for gi in G_by_unit.get(uid, []):
            g = GOLD[gi]
            if s < g[2] and e > g[1]:
                hitA = True; mA[gi] = 1
                if g[3] == c:
                    hitD = True; mD[gi] = 1
        ai = ad_idx[unit_ad[uid]]
        fpA[ai] += not hitA; fpD[ai] += not hitD
    return mA, mD, fpA, fpD

def eval_counts(run):
    fm, ff = run["fm"], run["ff"]; p = fm | ff
    A = np.zeros((NA, len(LEVELS), 4))
    bc = lambda arr: np.bincount(te_ad, weights=arr.astype(float), minlength=NA)
    for li, (yy, pp) in enumerate(((y_te, p), (ym_te, fm), (yf_te, ff))):
        for k, arr in enumerate((yy & pp, ~yy & pp, yy & ~pp, ~yy & ~pp)):
            A[:, li, k] = bc(arr)
    ay, ap = bc(y_te) > 0, bc(p) > 0
    for k, arr in enumerate((ay & ap, ~ay & ap, ay & ~ap, ~ay & ~ap)):
        A[:, 3, k] = arr
    mA, mD, fpA, fpD = phrase_match(run["phr"])
    for li, m, fp in ((4, mA, fpA), (5, mD, fpD)):
        A[:, li, 0] = np.bincount(G_ad, weights=m, minlength=NA); A[:, li, 1] = fp; A[:, li, 2] = np.bincount(G_ad, weights=1 - m, minlength=NA)
    return A, mA, mD

SYSTEMS = {
    "A1 gender-coded lexicon": [RUN_A1],
    "A2 TF-IDF-weighted lexicon": [RUN_A2],
    "B TF-IDF + logistic regression": [RUN_B],
    f"C DistilBERT (mean of {len(RUNS_C)} seeds)": RUNS_C,
}
CNT, MATCH = {}, {}
for name, runs in SYSTEMS.items():
    ev = [eval_counts(r) for r in runs]
    CNT[name] = np.mean([e[0] for e in ev], axis=0); MATCH[name] = (np.mean([e[1] for e in ev], axis=0), np.mean([e[2] for e in ev], axis=0))

def metrics_arr(S):
    tp, fp, fn, tn = S[..., 0], S[..., 1], S[..., 2], S[..., 3]
    dv = lambda a, b: np.divide(a, b, out=np.full(np.shape(a), np.nan), where=np.asarray(b) > 0)
    p, r = dv(tp, tp + fp), dv(tp, tp + fn)
    return {"precision": p, "recall": r, "f1": dv(2 * tp, 2 * tp + fp + fn), "fpr": dv(fp, fp + tn)}

B_RES = PARAMS["bootstrap_resamples"]
IDX = np.random.RandomState(PARAMS["bootstrap_seed"]).randint(0, NA, (B_RES, NA))
BOOT = {n: np.stack([c[IDX[b]].sum(0) for b in range(B_RES)]) for n, c in CNT.items()}

rows = []
for n, c in CNT.items():
    pt, bt = metrics_arr(c.sum(0)), metrics_arr(BOOT[n])
    for li, lv in enumerate(LEVELS):
        for mn in ("precision", "recall", "f1", "fpr"):
            if mn == "fpr" and lv not in FPR_LEVELS:
                continue
            lo, hi = np.nanpercentile(bt[mn][:, li], [2.5, 97.5])
            rows.append({"system": n, "level": lv, "metric": mn, "value": float(pt[mn][li]), "ci_low": float(lo), "ci_high": float(hi)})
res = pd.DataFrame(rows)
res.to_csv(OUT / "results_long.csv", index=False, encoding="utf-8-sig")
wide = res.assign(txt=lambda d: d.apply(lambda r: f"{r.value:.3f} ({r.ci_low:.3f}-{r.ci_high:.3f})", axis=1)).pivot_table(index=["level", "system"], columns="metric", values="txt", aggfunc="first")[["precision", "recall", "f1", "fpr"]]
wide = wide.reindex([l for l in LEVELS], level=0)
display(wide)
wide.to_csv(OUT / "results_table.csv", encoding="utf-8-sig")

cnt_rows = [{"system": n, "level": lv, **dict(zip(("TP", "FP", "FN", "TN"), c.sum(0)[li].round(1)))} for n, c in CNT.items() for li, lv in enumerate(LEVELS)]
display(pd.DataFrame(cnt_rows))

test: 2,142 units, 400 ads, 184 silver I1/I2 spans (84 I1, 100 I2)


metric                                               precision               recall                   f1                  fpr
level      system                                                                                                            
unit       A1 gender-coded lexicon         1.000 (1.000-1.000)  0.981 (0.957-1.000)  0.990 (0.978-1.000)  0.000 (0.000-0.000)
           A2 TF-IDF-weighted lexicon      1.000 (1.000-1.000)  0.981 (0.957-1.000)  0.990 (0.978-1.000)  0.000 (0.000-0.000)
           B TF-IDF + logistic regression  0.983 (0.955-1.000)  0.735 (0.660-0.811)  0.841 (0.788-0.890)  0.001 (0.000-0.003)
           C DistilBERT (mean of 3 seeds)  0.932 (0.895-0.969)  0.914 (0.870-0.955)  0.923 (0.888-0.951)  0.005 (0.002-0.008)
unit_masc  A1 gender-coded lexicon         1.000 (1.000-1.000)  0.961 (0.913-1.000)  0.980 (0.955-1.000)  0.000 (0.000-0.000)
           A2 TF-IDF-weighted lexicon      1.000 (1.000-1.000)  0.961 (0.913-1.000)  0.980 (0.955-1.000)  0.000 (0.000-0.000)
           B TF-IDF + logistic regression  0.950 (0.892-1.000)  0.750 (0.662-0.845)  0.838 (0.773-0.899)  0.001 (0.000-0.003)
           C DistilBERT (mean of 3 seeds)  0.894 (0.829-0.947)  0.921 (0.857-0.976)  0.907 (0.857-0.947)  0.004 (0.002-0.007)
unit_fem   A1 gender-coded lexicon         0.989 (0.963-1.000)  1.000 (1.000-1.000)  0.994 (0.981-1.000)  0.000 (0.000-0.001)
           A2 TF-IDF-weighted lexicon      0.989 (0.963-1.000)  1.000 (1.000-1.000)  0.994 (0.981-1.000)  0.000 (0.000-0.001)
           B TF-IDF + logistic regression  0.985 (0.948-1.000)  0.722 (0.612-0.829)  0.833 (0.752-0.904)  0.000 (0.000-0.001)
           C DistilBERT (mean of 3 seeds)  0.931 (0.885-0.968)  0.904 (0.846-0.954)  0.917 (0.877-0.950)  0.003 (0.001-0.005)
ad         A1 gender-coded lexicon         1.000 (1.000-1.000)  1.000 (1.000-1.000)  1.000 (1.000-1.000)  0.000 (0.000-0.000)
           A2 TF-IDF-weighted lexicon      1.000 (1.000-1.000)  1.000 (1.000-1.000)  1.000 (1.000-1.000)  0.000 (0.000-0.000)
           B TF-IDF + logistic regression  1.000 (1.000-1.000)  0.826 (0.750-0.890)  0.905 (0.857-0.942)  0.000 (0.000-0.000)
           C DistilBERT (mean of 3 seeds)  0.969 (0.936-0.994)  0.951 (0.908-0.987)  0.960 (0.932-0.982)  0.011 (0.002-0.024)
phrase     A1 gender-coded lexicon         0.995 (0.982-1.000)  0.984 (0.963-1.000)  0.989 (0.977-0.998)                  NaN
           A2 TF-IDF-weighted lexicon      0.995 (0.982-1.000)  0.984 (0.963-1.000)  0.989 (0.977-0.998)                  NaN
           B TF-IDF + logistic regression  0.643 (0.576-0.718)  0.745 (0.667-0.819)  0.690 (0.629-0.751)                  NaN
           C DistilBERT (mean of 3 seeds)  0.785 (0.709-0.860)  0.716 (0.641-0.790)  0.749 (0.689-0.807)                  NaN
phrase_dir A1 gender-coded lexicon         0.995 (0.982-1.000)  0.984 (0.963-1.000)  0.989 (0.977-0.998)                  NaN
           A2 TF-IDF-weighted lexicon      0.995 (0.982-1.000)  0.984 (0.963-1.000)  0.989 (0.977-0.998)                  NaN
           B TF-IDF + logistic regression  0.637 (0.569-0.714)  0.745 (0.667-0.819)  0.687 (0.627-0.747)                  NaN
           C DistilBERT (mean of 3 seeds)  0.779 (0.704-0.853)  0.716 (0.641-0.790)  0.746 (0.687-0.802)                  NaN

,system,level,TP,FP,FN,TN
0,A1 gender-coded lexicon,unit,152.0,0.0,3.0,1987.0
1,A1 gender-coded lexicon,unit_masc,73.0,0.0,3.0,2066.0
2,A1 gender-coded lexicon,unit_fem,90.0,1.0,0.0,2051.0
3,A1 gender-coded lexicon,ad,109.0,0.0,0.0,291.0
4,A1 gender-coded lexicon,phrase,181.0,1.0,3.0,0.0
5,A1 gender-coded lexicon,phrase_dir,181.0,1.0,3.0,0.0
6,A2 TF-IDF-weighted lexicon,unit,152.0,0.0,3.0,1987.0
7,A2 TF-IDF-weighted lexicon,unit_masc,73.0,0.0,3.0,2066.0
8,A2 TF-IDF-weighted lexicon,unit_fem,90.0,1.0,0.0,2051.0
9,A2 TF-IDF-weighted lexicon,ad,109.0,0.0,0.0,291.0


## 6. Paired differences

Paired bootstrap differences (same resamples) for the comparisons of interest. An interval that excludes 0 is a difference that the 400-advertisement sample supports; an interval that includes 0 is not evidence of a difference.

In [9]:
names = list(CNT)
pairs = [(names[3], names[0]), (names[3], names[2]), (names[2], names[0]), (names[1], names[0])]
MB = {n: metrics_arr(BOOT[n]) for n in names}; MP = {n: metrics_arr(CNT[n].sum(0)) for n in names}
rows = []
for a, b in pairs:
    for lv in ("unit", "ad", "phrase"):
        li = LEVELS.index(lv)
        for mn in ("f1", "fpr", "recall"):
            if mn == "fpr" and lv not in FPR_LEVELS:
                continue
            d = MB[a][mn][:, li] - MB[b][mn][:, li]; lo, hi = np.nanpercentile(d, [2.5, 97.5])
            rows.append({"comparison": f"{a.split()[0]} minus {b.split()[0]}", "level": lv, "metric": mn, "difference": round(float(MP[a][mn][li] - MP[b][mn][li]), 4),
                         "ci_low": round(float(lo), 4), "ci_high": round(float(hi), 4), "interval_excludes_0": bool(lo > 0 or hi < 0)})
paired = pd.DataFrame(rows); display(paired)
paired.to_csv(OUT / "paired_differences.csv", index=False)

,comparison,level,metric,difference,ci_low,ci_high,interval_excludes_0
0,C minus A1,unit,f1,-0.0673,-0.1049,-0.0363,True
1,C minus A1,unit,fpr,0.0052,0.0025,0.0083,True
2,C minus A1,unit,recall,-0.0667,-0.1166,-0.0186,True
3,C minus A1,ad,f1,-0.0401,-0.0683,-0.0181,True
4,C minus A1,ad,fpr,0.0115,0.0023,0.0239,True
5,C minus A1,ad,recall,-0.0489,-0.0918,-0.0129,True
6,C minus A1,phrase,f1,-0.2403,-0.3001,-0.1814,True
7,C minus A1,phrase,recall,-0.2681,-0.3478,-0.1883,True
8,C minus B,unit,f1,0.0816,0.0385,0.1273,True
9,C minus B,unit,fpr,0.0042,0.0015,0.0072,True


## 7. Bias score (-1 to +1) per advertisement

Each system gives every test advertisement a score by the formula in section 2. The silver score uses silver span counts. We report agreement on the 400 advertisements and on the advertisements that carry silver coded wording. Advertisement labels follow the balance rule of [1] as recorded in our methods file, with a fourth class for no signal: *masculine-coded* (M > F), *feminine-coded* (F > M), *balanced* (M = F > 0), *no signal* (M = F = 0). For DistilBERT the seed-averaged evidence is used. Because A2, B and C use weights or probabilities, an exact tie is rare, so *balanced* is mostly a silver-side class; this penalises those systems on the four-class accuracy and is reported in the limitations.

In [10]:
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import f1_score
CLASSES = ["masculine-coded", "feminine-coded", "balanced", "no signal"]

def ad_sums(runs):
    return (np.mean([np.bincount(te_ad, weights=r["M"], minlength=NA) for r in runs], axis=0), np.mean([np.bincount(te_ad, weights=r["F"], minlength=NA) for r in runs], axis=0))
def score_of(M, F):
    tot = M + F; return np.where(tot > 1e-12, (M - F) / np.where(tot > 1e-12, tot, 1.0), 0.0)
def label_of(M, F):
    tot = M + F; s = score_of(M, F)
    return np.select([tot <= 1e-12, s > 1e-9, s < -1e-9], ["no signal", "masculine-coded", "feminine-coded"], "balanced")

Ms = np.bincount(G_ad, weights=[g[3] == "I1" for g in GOLD], minlength=NA); Fs = np.bincount(G_ad, weights=[g[3] == "I2" for g in GOLD], minlength=NA)
S_SIL, L_SIL = score_of(Ms, Fs), label_of(Ms, Fs)
SUB = (Ms + Fs) > 0
print("silver advertisement labels on the 400 test ads:", dict(Counter(L_SIL)), f"| ads with silver coded wording: {int(SUB.sum())}")

SC, LB = {}, {}
for n, runs in SYSTEMS.items():
    M, F = ad_sums(runs); SC[n], LB[n] = score_of(M, F), label_of(M, F)

def sc_metrics(s, l, ii):
    d = np.abs(s[ii] - S_SIL[ii]); m = SUB[ii]
    try: pr = pearsonr(s[ii][m], S_SIL[ii][m])[0]
    except Exception: pr = np.nan
    try: sp = spearmanr(s[ii][m], S_SIL[ii][m])[0]
    except Exception: sp = np.nan
    dirn = np.mean(np.sign(s[ii][m]) == np.sign(S_SIL[ii][m])) if m.any() else np.nan
    return {"MAE (all ads)": d.mean(), "Pearson r (ads with silver signal)": pr, "Spearman rho (ads with silver signal)": sp,
            "direction agreement (ads with silver signal)": dirn, "4-class accuracy": np.mean(l[ii] == L_SIL[ii]), "4-class macro-F1": f1_score(L_SIL[ii], l[ii], labels=CLASSES, average="macro", zero_division=0)}
rows = []
full = np.arange(NA)
for n in SYSTEMS:
    pt = sc_metrics(SC[n], LB[n], full)
    bs = pd.DataFrame([sc_metrics(SC[n], LB[n], IDX[b]) for b in range(B_RES)])
    for k, v in pt.items():
        rows.append({"system": n, "measure": k, "value": float(v), "ci_low": float(np.nanpercentile(bs[k], 2.5)), "ci_high": float(np.nanpercentile(bs[k], 97.5))})
bias_res = pd.DataFrame(rows)
bw = bias_res.assign(txt=lambda d: d.apply(lambda r: f"{r.value:.3f} ({r.ci_low:.3f}-{r.ci_high:.3f})", axis=1)).pivot_table(index="measure", columns="system", values="txt", aggfunc="first")[list(SYSTEMS)]
display(bw); bias_res.to_csv(OUT / "bias_score_agreement.csv", index=False)

ad_tbl = pd.DataFrame({"record_id": ad_ids, "route_v03": [route_of[a] for a in ad_ids], "silver_M": Ms, "silver_F": Fs, "silver_score": S_SIL, "silver_label": L_SIL})
for n in SYSTEMS:
    M, F = ad_sums(SYSTEMS[n]); k = n.split()[0]
    ad_tbl[f"{k}_M"], ad_tbl[f"{k}_F"], ad_tbl[f"{k}_score"], ad_tbl[f"{k}_label"] = M, F, SC[n], LB[n]
ad_tbl.to_csv(OUT / "ad_bias_scores_test.csv", index=False, encoding="utf-8-sig")
display(pd.crosstab(pd.Series(L_SIL, name="silver label"), pd.Series(LB[names[3]], name="DistilBERT label")))

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), sharex=True, sharey=True); jr = np.random.RandomState(1)
for ax, n in zip(axes, SYSTEMS):
    ax.scatter(S_SIL + jr.uniform(-.04, .04, NA), SC[n] + jr.uniform(-.04, .04, NA), s=10, alpha=.45)
    ax.plot([-1, 1], [-1, 1], lw=.8, color="grey"); ax.set_title(n, fontsize=9); ax.set_xlabel("silver bias score")
axes[0].set_ylabel("predicted bias score"); fig.tight_layout(); fig.savefig(OUT / "figure_bias_score_agreement.png", dpi=130); plt.show()

silver advertisement labels on the 400 test ads: {np.str_('no signal'): 291, np.str_('feminine-coded'): 53, np.str_('masculine-coded'): 42, np.str_('balanced'): 14} | ads with silver coded wording: 109


system,A1 gender-coded lexicon,A2 TF-IDF-weighted lexicon,B TF-IDF + logistic regression,C DistilBERT (mean of 3 seeds)
measure,,,,
4-class accuracy,0.993 (0.983-1.000),0.963 (0.943-0.980),0.910 (0.880-0.938),0.930 (0.905-0.953)
4-class macro-F1,0.965 (0.916-1.000),0.713 (0.691-0.731),0.641 (0.603-0.673),0.668 (0.635-0.694)
MAE (all ads),0.007 (0.000-0.018),0.012 (0.004-0.021),0.070 (0.046-0.095),0.049 (0.032-0.071)
Pearson r (ads with silver signal),0.983 (0.961-1.000),0.980 (0.957-0.999),0.833 (0.765-0.892),0.935 (0.895-0.966)
Spearman rho (ads with silver signal),0.982 (0.960-1.000),0.980 (0.955-0.999),0.820 (0.731-0.890),0.921 (0.867-0.957)
direction agreement (ads with silver signal),0.972 (0.937-1.000),0.862 (0.796-0.925),0.679 (0.590-0.762),0.798 (0.721-0.868)


DistilBERT label,feminine-coded,masculine-coded,no signal
silver label,,,
balanced,4,10,0
feminine-coded,49,2,2
masculine-coded,1,38,3
no signal,4,2,285


C:\Users\cyber\AppData\Local\Temp\ipykernel_41612\3752735191.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  axes[0].set_ylabel("predicted bias score"); fig.tight_layout(); fig.savefig(OUT / "figure_bias_score_agreement.png", dpi=130); plt.show()


## 8. Flagged phrases

For every system the flagged phrases (with the direction and the weight that triggered them) are written to `flagged_phrases_test.csv`. DistilBERT phrases use the 2-of-3 seed rule from section 4. Below are three test advertisements chosen from the silver labels (most masculine-coded, most feminine-coded, and a mixed one): each unit that any system or the silver labels flag is shown with the flagged phrase in double brackets.

In [11]:
def majority(runs, k=2):
    c, w = Counter(), {}
    for r in runs:
        for uid, s, e, d, wt in {(p[0], p[1], p[2], p[3]): p for p in r["phr"]}.values():
            c[(uid, s, e, d)] += 1; w.setdefault((uid, s, e, d), []).append(wt)
    return [(u, s, e, d, float(np.mean(w[(u, s, e, d)]))) for (u, s, e, d), n in c.items() if n >= k]

PHR = {n: (majority(r) if len(r) > 1 else r[0]["phr"]) for n, r in SYSTEMS.items()}
gold_span_set = [(g[0], g[1], g[2]) for g in GOLD]
rows = []
for n, ph in PHR.items():
    for uid, s, e, d, wt in sorted(set(ph), key=lambda x: (str(x[0]), x[1])):
        rows.append({"system": n, "record_id": unit_ad[uid], "unit_id": uid, "phrase": text_of[uid][s:e], "start": s, "end": e, "direction": "masculine-coded" if d == "I1" else "feminine-coded",
                     "weight": round(wt, 4), "overlaps_silver_span": any(u == uid and s < ge and e > gs for u, gs, ge in gold_span_set)})
flag_tbl = pd.DataFrame(rows); flag_tbl.to_csv(OUT / "flagged_phrases_test.csv", index=False, encoding="utf-8-sig")
print(flag_tbl.groupby("system").agg(phrases=("phrase", "size"), share_overlapping_silver=("overlaps_silver_span", "mean")).round(3))

def render(uid, ph):
    t = text_of[uid]; spans = sorted({(s, e) for u, s, e, d, w in ph if u == uid})
    out, pos = "", 0
    for s, e in spans:
        if s >= pos:
            out += t[pos:s] + "[[" + t[s:e] + "]]"; pos = e
    return out + t[pos:]

pick = []
sig = np.where(SUB)[0]
pick.append(ad_ids[sig[np.argmax(np.where(S_SIL[sig] > 0.99, Ms[sig] + Fs[sig], -1))]])
pick.append(ad_ids[sig[np.argmax(np.where(S_SIL[sig] < -0.99, Ms[sig] + Fs[sig], -1))]])
pick.append(ad_ids[sig[np.argmax(np.where(np.abs(S_SIL[sig]) < 0.2, Ms[sig] + Fs[sig], -1))]])
sil_ph = [(g[0], g[1], g[2], g[3], 1.0) for g in GOLD]
for a in pick:
    i = ad_idx[a]
    print(f"\n=== ad {a} | silver score {S_SIL[i]:+.2f} (I1 {int(Ms[i])}, I2 {int(Fs[i])}) | " + " | ".join(f"{n.split()[0]} {SC[n][i]:+.2f}" for n in SYSTEMS))
    for uid in test_units.loc[test_units["record_id"] == a, "unit_id"]:
        if any(u == uid for u, *_ in sil_ph) or any(u == uid for ph in PHR.values() for u, *_ in ph):
            print(f"  silver: {render(uid, sil_ph)}")
            for n in SYSTEMS:
                r = render(uid, PHR[n])
                if r != text_of[uid] or True:
                    print(f"    {n.split()[0]:>2}: {r}")

                                phrases  share_overlapping_silver
system                                                           
A1 gender-coded lexicon             182                     0.995
A2 TF-IDF-weighted lexicon          182                     0.995
B TF-IDF + logistic regression      215                     0.647
C DistilBERT (mean of 3 seeds)      152                     0.862

=== ad row-1295 | silver score +1.00 (I1 4, I2 0) | A1 +1.00 | A2 +1.00 | B +1.00 | C +1.00
  silver: Marketing Team [[Lead]]
    A1: Marketing Team [[Lead]]
    A2: Marketing Team [[Lead]]
     B: Marketing [[Team]] [[Lead]]
     C: Marketing Team [[Lead]]
  silver: Strong copywriting and editing skills, including the ability to write in different styles for different audience types and goals (i.e. demand, thought [[leadership]], sales enablement)
    A1: Strong copywriting and editing skills, including the ability to write in different styles for different audience types and goals (i.e. demand,

## 9. Error analysis and sensitivity checks

These checks show where the systems agree with the silver labels for the wrong reason or disagree for a defensible reason. They are descriptive and do not change any label.

In [12]:
def bounds(c):
    pt = metrics_arr(c.sum(0)); return pt

# 9a seen vs unseen silver wording
seen_words = {g[3] for u in pool_df["unit_id"] for g in gold_by_unit.get(u, [])}
seen_mask = np.array([g[4] in seen_words for g in GOLD])
rows = []
for n in SYSTEMS:
    mA = MATCH[n][0]
    for lab, m in (("wording seen in training/validation silver spans", seen_mask), ("wording not seen", ~seen_mask)):
        rows.append({"system": n, "silver wording": lab, "silver spans": int(m.sum()), "phrase recall": round(float(mA[m].mean()), 3) if m.any() else np.nan})
display(pd.DataFrame(rows).pivot(index="silver wording", columns="system", values="phrase recall")[list(SYSTEMS)])
print("unseen silver wording (test):", Counter(g[4] for g, s in zip(GOLD, seen_mask) if not s).most_common(12))

# 9b by component
def unit_cm(run, mask):
    p = run["fm"] | run["ff"]; return np.array([(y_te & p & mask).sum(), (~y_te & p & mask).sum(), (y_te & ~p & mask).sum(), (~y_te & ~p & mask).sum()], float)
rows = []
for comp in sorted(test_units["component"].unique()):
    mask = (test_units["component"] == comp).to_numpy()
    for n, runs in SYSTEMS.items():
        S = np.mean([unit_cm(r, mask) for r in runs], axis=0); tp, fp, fn, tn = S
        rows.append({"component": comp, "units": int(mask.sum()), "silver positive units": int((y_te & mask).sum()), "system": n, "precision": round(tp / (tp + fp), 3) if tp + fp else np.nan,
                     "recall": round(tp / (tp + fn), 3) if tp + fn else np.nan, "f1": round(2 * tp / (2 * tp + fp + fn), 3) if 2 * tp + fp + fn else np.nan})
display(pd.DataFrame(rows).pivot(index=["component", "units", "silver positive units"], columns="system", values="f1")[list(SYSTEMS)])

# 9c sensitivity: drop units whose only silver spans are possible role nouns (P candidates)
flagged_role = {}
for r in imp.itertuples():
    flagged_role.setdefault(r.unit_id, []).append(r.review_flag == "P_candidate_role_noun")
role_only = {u for u, v in flagged_role.items() if all(v)}
keep = ~test_units["unit_id"].isin(role_only).to_numpy()
rows = []
for n, runs in SYSTEMS.items():
    S = np.mean([unit_cm(r, keep) for r in runs], axis=0); tp, fp, fn, tn = S
    rows.append({"system": n, "units dropped (role-noun-only silver units)": int((~keep).sum()), "precision": round(tp / (tp + fp), 3), "recall": round(tp / (tp + fn), 3), "f1": round(2 * tp / (2 * tp + fp + fn), 3), "fpr": round(fp / (fp + tn), 4)})
display(pd.DataFrame(rows))

# 9d employer overlap
emp_pool = {employer_of[a] for a in pool_ads}
seen_ad = np.array([employer_of[a] in emp_pool for a in ad_ids])
rows = []
for lab, m in (("employer also in training pool", seen_ad), ("employer not in training pool", ~seen_ad)):
    for n in SYSTEMS:
        pt = metrics_arr(CNT[n][m].sum(0))
        rows.append({"subset": lab, "ads": int(m.sum()), "ads with silver implicit": int(SUB[m].sum()), "system": n, "unit F1": round(float(pt["f1"][0]), 3), "ad F1": round(float(pt["f1"][3]), 3), "phrase F1": round(float(pt["f1"][4]), 3)})
emp = pd.DataFrame(rows); display(emp.pivot(index=["subset", "ads", "ads with silver implicit"], columns="system", values="unit F1")[list(SYSTEMS)])
emp.to_csv(OUT / "employer_overlap_results.csv", index=False)

# 9e false-positive wording
for n in SYSTEMS:
    fp_words = Counter(text_of[u][s:e].lower() for u, s, e, d, w in set((p[0], p[1], p[2], p[3], 0) for p in PHR[n]) if not any(u == gu and s < ge and e > gs for gu, gs, ge in gold_span_set))
    print(f"{n}: phrases with no silver span -> {sum(fp_words.values())}; top wording: {fp_words.most_common(8)}")

system,A1 gender-coded lexicon,A2 TF-IDF-weighted lexicon,B TF-IDF + logistic regression,C DistilBERT (mean of 3 seeds)
silver wording,,,,
wording not seen,1.000,1.000,0.00,0.278
wording seen in training/validation silver spans,0.983,0.983,0.77,0.730


unseen silver wording (test): [('objective', 1), ('supports', 1), ('together', 1), ('connections', 1), ('commit', 1), ('honesty', 1)]


,,system,A1 gender-coded lexicon,A2 TF-IDF-weighted lexicon,B TF-IDF + logistic regression,C DistilBERT (mean of 3 seeds)
component,units,silver positive units,,,,
body_other,58,8,1.000,1.000,0.545,0.735
job_title,399,12,1.000,1.000,0.957,1.000
qualification,1685,135,0.989,0.989,0.844,0.928


,system,units dropped (role-noun-only silver units),precision,recall,f1,fpr
0,A1 gender-coded lexicon,19,1.000,0.978,0.989,0.0000
1,A2 TF-IDF-weighted lexicon,19,1.000,0.978,0.989,0.0000
2,B TF-IDF + logistic regression,19,0.980,0.721,0.831,0.0010
3,C DistilBERT (mean of 3 seeds),19,0.922,0.902,0.912,0.0052


,,system,A1 gender-coded lexicon,A2 TF-IDF-weighted lexicon,B TF-IDF + logistic regression,C DistilBERT (mean of 3 seeds)
subset,ads,ads with silver implicit,,,,
employer also in training pool,362,98,0.989,0.989,0.863,0.943
employer not in training pool,38,11,1.000,1.000,0.609,0.755


A1 gender-coded lexicon: phrases with no silver span -> 1; top wording: [('responsibility', 1)]
A2 TF-IDF-weighted lexicon: phrases with no silver span -> 1; top wording: [('responsibility', 1)]
B TF-IDF + logistic regression: phrases with no silver span -> 76; top wording: [('and', 27), ('team', 7), ('o', 3), ('positive', 3), ('data', 3), ('good', 2), ('prc', 2), ('interprets', 2)]
C DistilBERT (mean of 3 seeds): phrases with no silver span -> 21; top wording: [('resolution', 2), ('dispute', 2), ('and', 2), ('of', 1), ('critical', 1), ('accent', 1), ('language', 1), ('insights', 1)]


## 10. Validation checks

In [13]:
from sklearn.metrics import precision_recall_fscore_support
checks = []
def rec(name, ok, detail=""):
    checks.append({"check": name, "passed": bool(ok), "detail": detail})
rec("test sample has 400 distinct ads, all present in the corpus", len(set(test_ads)) == 400 and set(test_ads) <= set(units["record_id"]))
rec("test sample equals the Step 7 file", sorted(test_ads) == sorted(rd(S7 / "test_sample_400_ads.csv")["record_id"]))
rec("no test ad in training or validation", not (set(train_ads) | val_ads) & set(test_ads))
rec("train + val + test = all 1,500 ads", len(train_ads) + len(val_ads) + len(test_ads) == units["record_id"].nunique() == 1500, f"{len(train_ads)}+{len(val_ads)}+{len(test_ads)}")
rec("every silver I1/I2 span reproduces inside its unit text", bad_offsets == 0, f"{bad_offsets} mismatches")
rec("TF-IDF and logistic regression fitted on training units only", not (FIT_IDS_A | FIT_IDS_B) & (set(va_units["unit_id"]) | set(test_units["unit_id"])))
rec("lexicon matches reproduce inside unit text", all(text_of[u][m[0]:m[1]].lower() == m[3] for u in test_units["unit_id"] for m in LEX_M[u]))
rec("DistilBERT predictions cover every test unit for every seed", all(len(r["test_probs"]) == len(test_units) for r in RES_C), f"{len(RES_C)} seeds")
rec("unit-level counts sum to test units", all(abs(c.sum(0)[0].sum() - len(test_units)) < 1e-6 for c in CNT.values()))
rec("ad-level counts sum to 400", all(abs(c.sum(0)[3].sum() - 400) < 1e-6 for c in CNT.values()))
sk = precision_recall_fscore_support(y_te, RUN_A1["fm"] | RUN_A1["ff"], average="binary", zero_division=0)[:3]
mine = metrics_arr(CNT[names[0]].sum(0)); mine3 = [float(mine["precision"][0]), float(mine["recall"][0]), float(mine["f1"][0])]
rec("unit-level A1 P/R/F1 match scikit-learn", np.allclose(sk, mine3), f"sklearn {np.round(sk, 4)} vs notebook {np.round(mine3, 4)}")
rec("all bias scores lie in [-1, +1]", all(np.all(np.abs(v) <= 1 + 1e-9) for v in list(SC.values()) + [S_SIL]))
rec("flagged phrase offsets are valid", all(0 <= r.start < r.end <= len(text_of[r.unit_id]) for r in flag_tbl.itertuples()))
rec("results table has every system, level and metric", len(res) == 4 * (4 * 4 + 2 * 3), f"{len(res)} rows")
chk = pd.DataFrame(checks); display(chk)
chk.to_csv(OUT / "validation_checks.csv", index=False)
print(f"{int(chk['passed'].sum())} of {len(chk)} checks passed.")

summary = {"test_ads": NA, "test_units": len(test_units), "silver_spans_test": len(GOLD), "bias_score": "score = (M - F) / (M + F), 0 if M + F = 0",
           "tau": BEST_TAU, "logreg": {"C": B_C, "threshold": B_TH}, "distilbert": [{k: r[k] for k in ("seed", "best_epoch", "threshold", "val_f1_at_threshold")} for r in RES_C],
           "results": {n: {lv: {mn: float(res[(res.system == n) & (res.level == lv) & (res.metric == mn)].value.iloc[0]) for mn in ["precision", "recall", "f1"] + ([] if lv not in FPR_LEVELS else ["fpr"])} for lv in LEVELS} for n in SYSTEMS}}
json.dump(summary, open(OUT / "results_summary.json", "w"), indent=1)

,check,passed,detail
0,"test sample has 400 distinct ads, all present in the corpus",True,
1,test sample equals the Step 7 file,True,
2,no test ad in training or validation,True,
3,"train + val + test = all 1,500 ads",True,990+110+400
4,every silver I1/I2 span reproduces inside its unit text,True,0 mismatches
5,TF-IDF and logistic regression fitted on training units only,True,
6,lexicon matches reproduce inside unit text,True,
7,DistilBERT predictions cover every test unit for every seed,True,3 seeds
8,unit-level counts sum to test units,True,
9,ad-level counts sum to 400,True,


14 of 14 checks passed.


## 11. Interpretation and limitations

All numbers below measure agreement with the rule-derived (silver) I1/I2 labels on the 400-ad test subset. They are not accuracy against human judgement, and no human gold was used at this stage.

### 11.1 Findings

**Lexicon methods (A1, A2) agree almost perfectly with the silver labels, and this is circularity, not skill.** The silver I1/I2 spans came from the same Gaucher-style stem lists. On the training pool the lexicon reproduced 533 of 544 silver spans, and the only silver-only wording was `confidential*` (11 spans), which the lexicon deliberately excludes. A1 unit F1 was 0.990 with FPR 0.000, and ad-level F1 was 1.000. We therefore read the A rows as a check that the lexicon was implemented as the rules describe, and not as evidence that the lexicon detects implicit bias well. A2 equals A1 at unit and ad-detection level because the validation-tuned tau was 0.0. TF-IDF weighting did change the bias score: A2 direction agreement fell to 0.862 (A1 0.972) and 4-class macro-F1 to 0.713 (A1 0.965).

**TF-IDF + logistic regression (B) is precise but misses wording it has not seen.** B reached unit precision 0.983 and recall 0.735 (F1 0.841). On silver wording absent from the training and validation spans, B's recall was 0.00, against 0.77 for wording that was present. Its flagged phrases were weak: phrase F1 0.690, with 76 of 213 phrases unrelated to any silver span, mostly function words such as "and" (27 phrases). We found that sparse n-gram features correlate with the silver labels mainly through memorised vocabulary. The proposed method in this study tended to have an inordinately higher proportion of missed units for new wording than the other systems.

**Fine-tuned DistilBERT (C) trades precision for recall and generalises partly beyond seen wording.** Averaged over three seeds, C reached unit F1 0.923 (P 0.932, R 0.914), ad F1 0.960 and phrase F1 0.749. Against B it had higher unit F1 (+0.082, interval excludes 0) and higher unit recall (+0.179), at a higher unit FPR (+0.004, still 0.005 in absolute terms). Its recall on wording unseen in training was 0.278, against 0.000 for B, which suggests the pretrained encoder carries some lexical-semantic knowledge. This is a modest recovery and rests on only six unseen wordings in the test set, so it should not be generalised. Phrase F1 for C versus B did not differ reliably (+0.059, interval -0.024 to 0.140).

**Bias score.** C tracked the silver score better than B (Pearson 0.935 vs 0.833, MAE 0.049 vs 0.070) but below the lexicon (0.983, 0.008). The weakest class for all learned models was `balanced`: DistilBERT labelled 10 of the 14 silver-balanced ads as masculine-coded. Four-class macro-F1 was 0.668 for C and 0.641 for B, so the score is better read as a continuous gender-coding signal than as a four-way label. Our study suggests that a higher unit-level F1 is not associated with a better ad-level balance label. The proposed scoring may benefit from the unit-level probabilities without adversely impacting ad-level detection (C ad recall 0.951, FPR 0.011), but the balanced class needs a larger sample.

### 11.2 Comparison with the literature

The comparison is qualitative only, because the tasks, languages and reference labels differ and numbers are not transferable.

- Dikshit et al. [1] study gender bias in STEM job advertisements and describe a word-list and balance-rule approach. Our ad labels follow that balance rule as recorded in the project methods file; we did not verify its results against the full text.
- Hu et al. [4], Mihaljevic et al. [5] and Jiang et al. [8] use gendered word lists to flag and measure coded wording in advertisements. Our A1/A2 results are consistent with word lists being easy to apply and transparent, but we cannot say they are valid, since our reference was built from the same list.
- Everitt et al. [23] report that regex-style matching works for predictable terms but does not scale because of polysemy, and that fine-tuned models outperform prompting for implicit bias. Our B and C results agree in direction on one point only: learned models gave non-zero recall on some unseen wording, whereas the list cannot. We do not compare numbers.
- Frissen et al. [25] and Raza et al. [26] apply machine learning to bias detection in text. Their labels come from different annotation procedures, so we do not compare scores.
- Kapoor and Narayanan [14] show that leakage inflates reported ML performance. That concern applies here: silver labels derived from the lexicon, employer overlap and the lexicon's own spans all raise apparent scores.

**Literature gaps.** We found no verified 2022-2025 source in our base for (a) TF-IDF weighting of gender-coded words, (b) word-occlusion phrase attribution, (c) the bootstrap and FPR choices, or (d) the stratified sampling. These are stated as design choices of this study, not as literature-backed methods. Gaucher 2011 (word-list origin) and Sanh 2019 (DistilBERT) are outside the window.

### 11.3 Limitations and their likely impact

1. **Silver-label circularity.** A1 and A2 score near 1.0 because they share their source with the labels. Impact: the A rows cannot rank the lexicon against the learned models, and the C minus A1 differences (unit F1 -0.067) are biased against C. The true ordering is unknown until human gold exists.
2. **Role nouns in the silver set.** 106 of 728 silver spans (14.6%) were also Step 5 P candidates (for example, a role noun such as "lead"). Impact: some "implicit" labels may be non-biased uses. Dropping the 19 role-noun-only test units changed A1 F1 from 0.990 to 0.989 and C from 0.923 to 0.912, so the headline result is not driven by them, but the underlying labels are still unverified.
3. **`confidential*` disagreement.** Silver has 11 `confidential*` spans that the lexicon excludes. Impact: any such spans lower lexicon recall slightly; we did not count them separately on the test set.
4. **Employer overlap.** 362 of 400 test ads share an employer with the training pool. Impact: B and C look better on those ads (ad-level recall 0.863 and 0.943) than on the 38 unseen-employer ads (0.609 and 0.755). Expect lower performance on new employers.
5. **Small unseen-wording sample.** Only six unseen silver wordings occur in the test set. Impact: the generalisation claim for C is tentative.
6. **Single-unit input.** The classifier sees one sentence or requirement at a time. Impact: wording whose coding depends on the surrounding ad is missed or over-flagged; the full-advertisement context is not used.
7. **No human gold.** Impact: all results are agreement, not validity. The bias score measures the balance of coded wording, not harm or discriminatory effect.
8. **English-oriented model.** DistilBERT (uncased, English) was applied to text with some Filipino markers (47 units in the corpus). Impact: those units may be handled poorly. Not measured separately.
9. **The `balanced` class is rare** (14 of 400 test ads). Impact: macro-F1 is unstable and is the weakest metric here.
10. **Phrase attribution rules are not literature-backed.** The B phrase rule (top unigram contributions) and the C occlusion rule (minimum drop 0.10, kept if flagged in at least 2 of 3 seeds) were chosen by us. Impact: phrase F1 depends on these parameters; B's phrase F1 is depressed by function words that the rule selected.
11. **No trained span tagger.** Phrases come from unit classifiers, not a token-level model. Impact: phrase boundaries are approximate. A token-level model was built in an earlier step and is not compared here.
12. **Threshold tuning on a small validation set** (110 ads, 36 implicit units). Impact: thresholds differed across seeds (0.10 to 0.65) with near-identical validation F1, so the unit-level operating point is not stable.

Further and in-depth studies may be needed to confirm these results, especially regarding human-verified labels, employers outside the training pool, and context-aware classification.

## 12. References (IEEE)

Only sources verified in `literature-support-for-methods.md` are used for claims; abstract-level reading is stated where it applies.

[1] M. Dikshit, H. Bouamor, and N. Habash, "Investigating gender bias in STEM job advertisements," in *Proc. 5th Workshop Gender Bias Natural Lang. Process. (GeBNLP)*, 2024, pp. 179-189, doi: 10.18653/v1/2024.gebnlp-1.11.

[4] S. Hu *et al.*, "Balancing gender bias in job advertisements with text-level bias mitigation," *Front. Big Data*, vol. 5, Art. no. 805713, 2022, doi: 10.3389/fdata.2022.805713.

[5] H. Mihaljevic, I. Muller, K. Dill, and A. Yollu-Tok, "Towards gender-inclusive job postings: A data-driven comparison of augmented writing technologies," *PLOS ONE*, vol. 17, no. 9, Art. no. e0274312, Sep. 2022, doi: 10.1371/journal.pone.0274312.

[8] T. Jiang, Y. Li, S. Fu, and Y. Chen, "Creating a Chinese gender lexicon for detecting gendered wording in job advertisements," *Inf. Process. Manage.*, vol. 60, no. 5, Art. no. 103424, Sep. 2023, doi: 10.1016/j.ipm.2023.103424.

[14] S. Kapoor and A. Narayanan, "Leakage and the reproducibility crisis in machine-learning-based science," *Patterns*, vol. 4, no. 9, Art. no. 100804, Sep. 2023, doi: 10.1016/j.patter.2023.100804.

[23] T. Everitt, P. Ryan, B. Davis, and K. J. Adebayo, "Large language models for detecting bias in job descriptions," *Italian J. Comput. Linguistics*, vol. 10, no. 2, p. 79, 2024, doi: 10.17454/ijcol102.05.

[25] R. Frissen, K. J. Adebayo, and R. Nanda, "A machine learning approach to recognize bias and discrimination in job advertisements," *AI & Society*, vol. 38, pp. 1025-1038, 2023, doi: 10.1007/s00146-022-01574-0.

[26] S. Raza, M. Garg, D. J. Reji, S. R. Bashir, and C. Ding, "Nbias: A natural language processing framework for BIAS identification in text," *Expert Syst. Appl.*, 2024, Art. no. 121, ScienceDirect pii S0957417423020444 (DOI to be completed from the publisher page).

*Outside the 2022-2025 window, cited only to identify a resource:* D. Gaucher, J. Friesen, and A. C. Kay, "Evidence that gendered wording in job advertisements exists and sustains gender inequality," *J. Pers. Soc. Psychol.*, 2011, doi: 10.1037/a0022530 (word-list origin; the project uses the Gender Decoder implementation, not checked against the paper's table). V. Sanh *et al.*, DistilBERT, 2019 (not verified in this session).